<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/IOAI-Logo.png?raw=1" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/Radar.ipynb)

# Radar

## 1. Problem Description

Radar is a key technology in wireless communication, with widespread applications such as self-driving cars. It typically involves an antenna that transmits specific signals and receives their reflections from objects in the environment. By processing these signals, the system determines the angular direction, distance, and velocity of target objects.

In real-world applications, radar signal processing is challenging due to noise and reflections from non-target objects in the surroundings. For example, when attempting to detect pedestrians, the radar may also receive reflections from trees or other background objects, which can degrade accuracy. Your task is to use AI to analyze the signals received by the radar and identify the presence of a human at each position.

In this task, we provide an **indoor radar experiment dataset**, and your objective is to develop a model that performs **radar semantic segmentation**.


## 2. Dataset

To measure objects surrounding a radar, the following key parameters are used:

- **Range**: The straight-line distance between the radar and an object.
- **Azimuth**: The horizontal angle (left to right) between the radar and the object.
- **Elevation**: The vertical angle (up or down) of the object relative to the radar.
- **Velocity**: The speed at which the object is moving toward or away from the radar.

<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/Radar%20Fig%201.png?raw=1" width="300">


The radar data is processed into multiple **heatmaps**, each encoding the **received signal strength** at various positions and directions.

- **Static heatmaps** emphasize reflections from **stationary** objects.
- **Dynamic heatmaps** highlight changes caused by **moving** objects.

When no object is present at a specific location, the signal consists mostly of background noise and appears weak. In contrast, reflections from an object increase signal intensity, enabling detection of the object.

For example, the **static range-azimuth heatmap** represents signal strength across different distances (**range**) and horizontal angles (**azimuth**), mainly reflected by stationary objects.

Each sample in the dataset is stored in a `.mat.pt` file as a tensor of shape $7 \times 50 \times 181$, where:

- 7 is the number of maps (6 heatmaps + 1 semantic label map),
- 50 represents range bins (distance),
- 181 represents angular or velocity bins, covering angles from \-90° to \+90° in either the horizontal or vertical plane. You can assume that the velocity bins are also remapped from \-90° to \+90° for visualization consistency.
- each heatmap intensity value is normalized to [0, 1], representing received signal strength.

The 6 heatmaps are structured as follows:

- **Index 0**: Static range-azimuth heatmap
- **Index 1**: Dynamic range-azimuth heatmap
- **Index 2**: Static range-elevation heatmap
- **Index 3**: Dynamic range-elevation heatmap
- **Index 4**: Static range-velocity heatmap
- **Index 5**: Dynamic range-velocity heatmap

All values in heatmaps are **normalized**, so no unit conversion is required.

The **map at Index 6** is the semantic label map, stored in range-azimuth format.

- **-1**: Background (no target)
- **0**: Suitcase
- **1**: Chair
- **2**: Human
- **3**: Wall

This is the visualization of 1.mat.pt in training_set:

<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/Radar%20Fig%202.png?raw=1" width="675">

Here is part of a sample from the dataset:

<img src="https://github.com/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Radar/figs/Radar%20Fig%203.png?raw=1" width="675">


Data scale: 1800 samples in the training set, 500 samples in the validation set, and 500 samples in the test set.

## 3\. Task

Your task is to develop a model that takes the **first six heatmaps** (indices 0 to 5) as input, and predicts the **semantic label map** (index 6) as the output. The goal is to accurately identify what the target is(-1 to 3) at each location in the radar’s field of view.

1. **Input**: A tensor of shape $6 \times 50 \times 181$, representing six radar heatmaps.
2. **Output**: A tensor of shape $50 \times 181$, representing the target semantic label map.


## 4\. Submission

Please submit a file named `submission.ipynb`. The output is a zip file named "submission.zip", which contains two tables `submission_val.csv` and `submission_test.csv` corresponding to the prediction results of the validation set and the test set respectively.

**Note:** The output table should have a header, the data in the table is not the actual solved data, it is only used as an example of the submission format.

| filename | pixel_0 | pixel_1 | ... | pixel_9049 |
| :------: | :-----: | ------- | --- | ---------- |
| 1.mat.pt |   -1    | -1      | ... | -1         |
|   ...    |   ...   | ...     | ... | ...        |

## 5\. Score

The score is based on the **accuracy of label recognition**. Correctly identifying target points is weighted more heavily than correctly identifying background points.

### Scoring Criteria:

* Each correctly identified **background pixel** earns **1 point**.

* Each correctly identified **non-background pixel** earns **50 points**.

* The final score is normalized to a **0-1 point** by comparing it to the maximum possible score.

### Formula：
$$
Score = \frac{|C_{0,correct}| \times 1 + |C_{1,correct}| \times bonus}{|C_0| \times 1 + |C_1| \times bonus}
$$
where:

$$
\begin{aligned}
I &= \{1, 2, \dots, 50\times 181\}\\
C_0 &= \{i \in I \mid y_i = -1\}\\
C_1 &= \{i \in I \mid y_i \neq -1\}\\
C_{0,correct} &= \{i \in C_0 \mid p_i = y_i\}\\
C_{1,correct} &= \{i \in C_1 \mid p_i = y_i\}\\
\end{aligned}
$$


### Example

For a $3\times3$ heatmap, assume the Ground Truth is:

$$
\begin{bmatrix}
-1 & -1 & -1 \\
1 & 2 & 3 \\
-1 & -1 & -1
\end{bmatrix}
$$

The intenteded result is:

$$
\begin{bmatrix}
-1 & 1 & -1 \\
-1 & 2 & -1 \\
-1 & 3 & -1
\end{bmatrix}
$$

Then there are four correctly identified `-1` and one correctly identified `2`. Your score is 4 + 50 = 54 points. The maximum possible score is 6 + 50 * 3 = 156, that is, the score for six background pixels and three non-background pixels. Your normalized score is 54 / 156 = 0.346.

$$
Score = \frac{4 \times 1 + 1 \times 50}{6 \times 1 + 3 \times 50}=0.346
$$

## 6. Baseline and Training Set

- Below you can find the baseline solution.
- The dataset is in `training_set` folder.
- The highest score by the Scientific Committee for this task is 0.90 in Leaderboard B, this score is used for score unification.
- The baseline score by the Scientific Committee for this task is 0.67 in Leaderboard B, this score is used for score unification.

### Data Loading

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
!git clone --depth 1 https://github.com/IOAI-official/IOAI-2025.git
import os
from pathlib import Path

base_path = Path("/content/IOAI-2025/Individual-Contest/Radar/training_set")

if base_path.exists():
    files = list(base_path.glob("*.mat.pt"))
    print(f"Yes: {len(files)}")
else:
    print("Nooo.")

Cloning into 'IOAI-2025'...
remote: Enumerating objects: 20169, done.
remote: Counting objects: 100% (20169/20169), done.
remote: Compressing objects: 100% (19365/19365), done.
remote: Total 20169 (delta 788), reused 20142 (delta 786), pack-reused 0 (from 0)
Receiving objects: 100% (20169/20169), 2.73 GiB | 37.99 MiB/s, done.
Resolving deltas: 100% (788/788), done.
Updating files: 100% (20080/20080), done.
Yes: 1800


Here im doing augmentation with albumention library

In [ ]:
import albumentations as A
import cv2
import torchvision
from torchvision import transforms
from torchvision.transforms import ToTensor
transform = A.Compose([
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(std_range=(0.1, 0.2), p=0.2),
    A.ToTensorV2()
])

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class CustomDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]
        labels = data[6]

        images = images.float()
        labels = labels.long()
        labels = labels + 1

        images = images.permute(1, 2, 0).numpy()
        labels = labels.numpy()
        #Here is augmentation part
        if self.transform:
          augmented = self.transform(image=images, mask=labels)
          images = augmented['image']
          labels = augmented['mask']

        if images.shape[-1] == 6:
          images = torch.tensor(images)
          images = images.permute(2, 0, 1)

        return images, labels, self.file_names[idx]

class CustomDataset_test(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]

        images = images.float()

        if self.transform:
            images = self.transform(images)

        return images, self.file_names[idx]

def generate_file_paths(base_path):
    file_paths = []
    for frame in os.listdir(base_path):
        frame_path = os.path.join(base_path, frame)
        if frame_path.endswith('.mat.pt'):
            file_paths.append(frame_path)
    return [path for path in file_paths if os.path.exists(path)]

def load_data(base_path, batch_size=4, num_workers=2, test_size=0.2):
    file_paths = generate_file_paths(base_path)

    train_paths, test_paths = train_test_split(file_paths, test_size=test_size, random_state=42)

    train_dataset = CustomDataset(file_paths=train_paths, transform=transform) #transform for augmentation
    test_dataset = CustomDataset(file_paths=test_paths)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=True
    )

    return train_loader, test_loader

### Model Definition and Training

that is my first model called MyModel

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()

        self.relu = nn.ReLU()

        self.conv1 = nn.Conv2d(6, 64, 3, 1, 1)
        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv3 = nn.Conv2d(64, 32, 3, 1, 1)
        self.conv4 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv5 = nn.Conv2d(32, 16, 3, 1, 1)
        self.conv6 = nn.Conv2d(16, 16, 3, 1, 1)
        self.conv7 = nn.Conv2d(16, 16, 3, 1, 1)
        self.conv8 = nn.Conv2d(16, 32, 3, 1, 1)
        self.conv9 = nn.Conv2d(32, 32, 3, 1, 1)
        self.conv10 = nn.Conv2d(32, 64, 3, 1, 1)
        self.conv11 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv12 = nn.Conv2d(64, 64, 3, 1, 1)

        self.b1 = nn.BatchNorm2d(64)
        self.b2 = nn.BatchNorm2d(64)

        self.b3 = nn.BatchNorm2d(32)
        self.b4 = nn.BatchNorm2d(32)

        self.b5 = nn.BatchNorm2d(16)
        self.b6 = nn.BatchNorm2d(16)

        self.b7 = nn.BatchNorm2d(16)
        self.b8 = nn.BatchNorm2d(32)

        self.b9 = nn.BatchNorm2d(32)
        self.b10 = nn.BatchNorm2d(64)

        self.b11 = nn.BatchNorm2d(64)
        self.b12 = nn.BatchNorm2d(64)

        self.pool1 = nn.MaxPool2d(2,2, return_indices=True)
        self.pool2 = nn.MaxPool2d(2,2, return_indices=True)
        self.pool3 = nn.MaxPool2d(2,2, return_indices=True)

        self.unpool1 = nn.MaxUnpool2d(2,2)
        self.unpool2 = nn.MaxUnpool2d(2,2)
        self.unpool3 = nn.MaxUnpool2d(2,2)

    def forward(self, x):
        x = self.relu(self.b1(self.conv1(x)))
        x = self.relu(self.b2(self.conv2(x)))
        size1 = x.size()
        x, ind1 = self.pool1(x)

        x = self.relu(self.b3(self.conv3(x)))
        x = self.relu(self.b4(self.conv4(x)))
        size2 = x.size()
        x, ind2 = self.pool2(x)

        x = self.relu(self.b5(self.conv5(x)))
        x = self.relu(self.b6(self.conv6(x)))
        size3 = x.size()
        x, ind3 = self.pool3(x)

        x = self.unpool3(x, ind3, output_size=size3)
        x = self.relu(self.b7(self.conv7(x)))

        x = self.relu(self.b8(self.conv8(x)))

        x = self.unpool2(x, ind2, output_size=size2)
        x = self.relu(self.b9(self.conv9(x)))
        x = self.relu(self.b10(self.conv10(x)))

        x = self.unpool1(x, ind1, output_size=size1)
        x = self.relu(self.b11(self.conv11(x)))
        x = self.relu(self.b12(self.conv12(x)))

        return x

and that is my second model, i tried to write kind of SegNet

In [ ]:
#simple class for conv -> batchnorm -> relu, so that class is our one encoder layer
class ConvRelu(nn.Module):
  def __init__(self, in_c, out_c, kernel_size = 3, padding = 1):
    super().__init__()
    self.conv = nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding)
    self.bn = nn.BatchNorm2d(out_c)
    self.relu = nn.ReLU(inplace=True)

  def forward(self, x):
    return self.relu(self.bn(self.conv(x)))


# this is class for one encoder block, in one encoder block we have count = depth encoder layers(ConvRelu)
class EncoderBlock(nn.Module):
  def __init__(self, in_c, out_c, depth = 2, kernel_size=3, padding=1):
    super().__init__()
    self.layers = nn.ModuleList()
    for i in range(depth):
      self.layers.append(ConvRelu(in_c if i == 0 else out_c, out_c, kernel_size=kernel_size, padding=padding))
    self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2, return_indices = True)

  def forward(self, x):
    for layer in self.layers:
      x = layer(x)
    size = x.size()
    x, ind = self.pool(x)
    return x, ind, size


#Decoder block contains count=depth decoder layers(ConvRelu)
class DecoderBlock(nn.Module):
  def __init__(self, in_c, out_c, depth = 2, kernel_size = 3, padding = 1, classification=False):
    super().__init__()
    self.unpool = nn.MaxUnpool2d(kernel_size = 2, stride = 2)
    self.layers = nn.ModuleList()
    for i in range(depth):
      #1. i == 0(first iteration) -> layers.append(ConvRelu(in_c -> in_c))
      #2. i != 0 and classification = False -> layers.append(ConvRelu(in_c -> out_c))
      #3. i!=0 and classification=True -> layers.append(Conv2d(in_c -> out_c))

      if i == depth-1 and classification==True:
        self.layers.append(nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding))
      elif i == depth-1 and classification==False:
        self.layers.append(ConvRelu(in_c, out_c, kernel_size=kernel_size, padding=padding))
      else:
        self.layers.append(ConvRelu(in_c, in_c, kernel_size=kernel_size, padding=padding))

  def forward(self, x, ind, size):
    x = self.unpool(x, ind, output_size=size)
    for layer in self.layers:
      x = layer(x)
    return x


class SegNet(nn.Module):
  def __init__(self, in_channels=6, out_channels=5, features = 64):
    super().__init__()
    #Encoders
    self.enc0 = EncoderBlock(in_channels, features)
    # self.enc1 = EncoderBlock(features, 2*features)
    self.enc2 = EncoderBlock(features, 2*features, depth=3)
    # self.enc3 = EncoderBlock(4*features, 8*features, depth = 3)

    #Bottleneck
    self.bn0 = EncoderBlock(2*features, 2*features, depth=3)
    self.bn1 = DecoderBlock(2*features, 2*features, depth = 3)

    #Decoders
    self.dec0 = DecoderBlock(2*features, features, depth = 3)
    # self.dec1 = DecoderBlock(4*features, 2*features, depth = 3)
    # self.dec2 = DecoderBlock(2*features, features)
    self.dec3 = DecoderBlock(features, out_channels, classification=True)

  def forward(self, x):
    x, ind0, size0 = self.enc0(x)
    # x, ind1, size1 = self.enc1(x)
    x, ind2, size2 = self.enc2(x)
    #x, ind3, size3 = self.enc3(x)

    x, indb, sizeb = self.bn0(x)
    x = self.bn1(x, indb, sizeb)

    x = self.dec0(x, ind2, size2)
    # x = self.dec1(x, ind2, size2)
    # x = self.dec2(x, ind1, size1)
    x = self.dec3(x, ind0, size0)

    return x


# and this is my implementation for UNet with depth = 3

in the original UNet there is depth = 4, but h and w of my data is pretty small, so i decided to use just 3 decoders and encoders blocks

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
#Now let's write classes for UNet, becouse how said ynker Tigran, 'try Unet'
#it is basic class for every Decoder and encoder block
class DoubleConvRelu(nn.Module):
  def __init__(self, in_c, out_c, kernel_size=3, padding=1):
    super().__init__()
    self.dp1 = nn.Dropout(0.5)
    self.dp2 = nn.Dropout(0.5)
    self.conv1 = nn.Conv2d(in_channels = in_c, out_channels = out_c, kernel_size=kernel_size, padding=padding)
    self.conv2 = nn.Conv2d(in_channels=out_c, out_channels=out_c, kernel_size=kernel_size, padding=padding)
    self.bn1 = nn.BatchNorm2d(out_c)
    self.bn2 = nn.BatchNorm2d(out_c)
    self.relu = nn.ReLU(inplace=True)

  def forward(self, x):
    # x = self.dp1(self.relu(self.bn1(self.conv1(x))))
    # x = self.dp2(self.relu(self.bn2(self.conv2(x))))
    x = self.relu(self.bn1(self.conv1(x)))
    x = self.relu(self.bn2(self.conv2(x)))
    return x


class UNet(nn.Module):
  def __init__(self, in_c, out_c, features=64, kernel_size=3, padding=1):
    super().__init__()
    self.down1 = DoubleConvRelu(in_c, features)
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)
    self.down2 = DoubleConvRelu(features, 2*features)
    self.down3 = DoubleConvRelu(2*features, 4*features)
    self.bottle = DoubleConvRelu(4*features, 8*features)
    self.upsample1 = nn.ConvTranspose2d(in_channels = 8*features, out_channels = 4*features, kernel_size=2, stride=2)
    self.up1 = DoubleConvRelu(8*features, 4*features)
    self.upsample2 = nn.ConvTranspose2d(4*features, 2*features, kernel_size=2, stride=2)
    self.up2 = DoubleConvRelu(4*features, 2*features)
    self.upsample3 = nn.ConvTranspose2d(2*features, features, kernel_size=2, stride=2)
    self.up3 = DoubleConvRelu(2*features, features)
    self.final = nn.Conv2d(features, out_c, kernel_size=1, padding=0)

  def forward(self, x):
    enc1 = self.down1(x)
    enc_pool1, ind1 = self.pool(enc1)
    size1 = enc1.size()                       #6x50x181 -> 25, 90 -> 12, 45
    enc2 = self.down2(enc_pool1)
    enc_pool2, ind2 = self.pool(enc2)
    size2 = enc2.size()
    enc3 = self.down3(enc_pool2)
    enc_pool3, ind3 = self.pool(enc3)
    size3 = enc3.size()

    bottle = self.bottle(enc_pool3)
    bottle_upsample = self.upsample1(bottle, output_size=size3)

    dec1 = self.up1(torch.cat([enc3, bottle_upsample], dim=1))
    dec_unpool1 = self.upsample2(dec1, output_size=size2)
    dec2 = self.up2(torch.cat([enc2, dec_unpool1], dim=1))
    dec_unpool2 = self.upsample3(dec2, output_size=size1)
    dec3 = self.up3(torch.cat([enc1, dec_unpool2], dim=1))
    res = self.final(dec3)

    return res

# Training part

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

def train(model, train_loader, test_loader, optimizer, criterion, scheduler, num_epochs=100):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0

        for images, labels, _ in train_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels

            outputs = model(images)
            outputs = outputs.view(outputs.size(0), outputs.size(1), -1)  # [B, C, H*W]
            labels = labels.view(labels.size(0), -1)  # [B, H*W]
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            batch_count += 1

        avg_train_loss = epoch_loss / batch_count
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        val_batch_count = 0

        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels

                outputs = model(images)
                outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
                labels = labels.view(labels.size(0), -1)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                val_batch_count += 1

        avg_val_loss = val_loss / val_batch_count
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)   #i added scheduler, vor changing lr na xadu


        if (epoch+1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}')

    return train_losses, val_losses
TRAIN_PATH = "./"
# The training set is deployed automatically in the testing machine.
# You notebook can access the TRAIN_PATH even if you do not mount it along with notebook.
data_path = "/content/IOAI-2025/Individual-Contest/Radar/training_set"

train_loader, test_loader = load_data(
    base_path=data_path,
    batch_size=4,
    num_workers=2,
    test_size=0.2
)

model = UNet(6, 5)
if torch.cuda.is_available():
    model = model.cuda()


weights = torch.tensor([1, 50, 50, 50, 50], dtype=torch.float32).cuda()   #Here is most important part of this problem
criterion = nn.CrossEntropyLoss(weight=weights)    #and also here
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=3
)

train_losses, val_losses = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler = scheduler,
    num_epochs=40
)

Epoch [2/40], Train Loss: 0.1041, Val Loss: 0.1135
Epoch [4/40], Train Loss: 0.0763, Val Loss: 0.0842
Epoch [6/40], Train Loss: 0.0681, Val Loss: 0.0662
Epoch [8/40], Train Loss: 0.0596, Val Loss: 0.0660
Epoch [10/40], Train Loss: 0.0565, Val Loss: 0.0642
Epoch [12/40], Train Loss: 0.0659, Val Loss: 0.0619
Epoch [14/40], Train Loss: 0.0495, Val Loss: 0.0581
Epoch [16/40], Train Loss: 0.0475, Val Loss: 0.0608
Epoch [18/40], Train Loss: 0.0440, Val Loss: 0.0618
Epoch [20/40], Train Loss: 0.0403, Val Loss: 0.0551
Epoch [22/40], Train Loss: 0.0363, Val Loss: 0.0570
Epoch [24/40], Train Loss: 0.0340, Val Loss: 0.0611
Epoch [26/40], Train Loss: 0.0326, Val Loss: 0.0631
Epoch [28/40], Train Loss: 0.0324, Val Loss: 0.0633
Epoch [30/40], Train Loss: 0.0319, Val Loss: 0.0634
Epoch [32/40], Train Loss: 0.0319, Val Loss: 0.0638
Epoch [34/40], Train Loss: 0.0318, Val Loss: 0.0633
Epoch [36/40], Train Loss: 0.0318, Val Loss: 0.0635
Epoch [38/40], Train Loss: 0.0316, Val Loss: 0.0635
Epoch [40/40], T

### Generate CSV for Submission

that part was written by IOAI comitete

In [ ]:
# -*- coding: utf-8 -*-
import os
import glob
import re
import zipfile
import pandas as pd
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def natural_key(s):
    s = os.path.basename(s)
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def ensure_str(fname):
    if isinstance(fname, bytes):
        return fname.decode('utf-8')
    if isinstance(fname, torch.Tensor):
        try:
            return fname.item() if fname.numel() == 1 else fname.tolist()
        except:
            return str(fname)
    return str(fname)

if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH")
else:
    DATA_PATH = "/content/IOAI-2025/Individual-Contest/Radar/Solution"

val_dir = os.path.join(DATA_PATH, "validation_set")
test_dir = os.path.join(DATA_PATH, "test_set")

val_paths = sorted(glob.glob(os.path.join(val_dir, "*.mat.pt")), key=natural_key)
test_paths = sorted(glob.glob(os.path.join(test_dir, "*.mat.pt")), key=natural_key)

print(f"Found files in validation_set: {len(val_paths)}")
print(f"Found files in test_set: {len(test_paths)}")

if len(val_paths) == 0:
    raise FileNotFoundError(f"validation_set is empty or dir is incorrect: {val_dir}")
if len(test_paths) == 0:
    print("Warning: test_set is empty. continue, but submission_test.csv will be empty.")

val_dataset = CustomDataset_test(file_paths=val_paths)
test_dataset = CustomDataset_test(file_paths=test_paths)

val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)


def run_inference(model, data_loader):
    model.eval()
    predictions = []
    filenames = []

    with torch.no_grad():
        for batch in data_loader:

            if isinstance(batch, (list, tuple)) and len(batch) >= 2:
                images, file_names = batch[0], batch[1]
            else:

                images = batch[0] if isinstance(batch, (list, tuple)) else batch
                file_names = None

            images = images.to(device, non_blocking=True)

            outputs = model(images)

            if outputs.dim() == 4:
                preds = torch.argmax(outputs, dim=1)  # (B, H, W)
            elif outputs.dim() == 3:

                preds = torch.argmax(outputs, dim=1)
            else:
                raise ValueError(f"unsuported form outputs: {outputs.shape}")
            preds = preds - 1

            if file_names is None:

                if hasattr(data_loader.dataset, "file_paths"):
                    start_idx = len(filenames)
                    end_idx = start_idx + preds.size(0)
                    batch_file_paths = data_loader.dataset.file_paths[start_idx:end_idx]
                    batch_names = [os.path.basename(p) for p in batch_file_paths]
                else:
                    batch_names = [f"sample_{i}" for i in range(len(preds))]
            else:
                if isinstance(file_names, (list, tuple)):
                    batch_names = [ensure_str(fn) for fn in file_names]
                else:
                    try:
                        batch_names = [ensure_str(fn) for fn in file_names.tolist()]
                    except:
                        batch_names = [str(fn) for fn in file_names]

                batch_names = [os.path.basename(n) for n in batch_names]

            for i in range(preds.size(0)):
                flat = preds[i].cpu().numpy().flatten().astype(int)
                predictions.append(flat)
                filenames.append(batch_names[i])

    return predictions, filenames

val_predictions, val_filenames = run_inference(model, val_loader)
print(f"Predictions on Val: {len(val_predictions)}, files: {len(val_filenames)}")

if len(val_predictions) != len(val_paths):
    print("Warning: something is very bad")

val_df = pd.DataFrame(val_predictions)
val_df.columns = [f'pixel_{i}' for i in range(val_df.shape[1])]
val_df.insert(0, 'filename', val_filenames)
val_df.to_csv('submission_val.csv', index=False)
print("Saved submission_val.csv, shape:", val_df.shape)

if len(test_paths) > 0:
    test_predictions, test_filenames = run_inference(model, test_loader)
    print(f"Predictions on test: {len(test_predictions)}, files: {len(test_filenames)}")
    test_df = pd.DataFrame(test_predictions)
    test_df.columns = [f'pixel_{i}' for i in range(test_df.shape[1])]
    test_df.insert(0, 'filename', test_filenames)
    test_df.to_csv('submission_test.csv', index=False)
    print("Saved submission_test.csv, shape:", test_df.shape)
else:
    empty = pd.DataFrame(columns=['filename'] + [f'pixel_{i}' for i in range(50*181)])
    empty.to_csv('submission_test.csv', index=False)
    print("created empty submission_test.csv (test_set is empty).")

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('submission_test.csv')

print("in submission.zip is only submission_test.csv")

print("Thats all Brother Sergo, go sleep")

Found files in validation_set: 500
Found files in test_set: 500
Predictions on Val: 500, files: 500
Saved submission_val.csv, shape: (500, 9051)
Predictions on test: 500, files: 500
Saved submission_test.csv, shape: (500, 9051)
in submission.zip is only submission_test.csv
Thats all Brother Sergo, go sleep
